# einops-repeat-broadcast composite — cx7: every-ray every-triangle (NR, NT, 3) via two repeats

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `einops-repeat`, `einops-repeat-broadcast`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "einops-repeat-broadcast"
DD_ATOM_IDS = ["einops-repeat", "einops-repeat-broadcast"]
DD_SUBTOPICS = ["Einops: Repeat", "Einops: Repeat-as-broadcast"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## How these two atoms compose

ARENA ray-tracing constantly builds `(NR, NT, ...)` tensors — one slot per (ray, triangle) pair. The free way is two `einops.repeat` calls, both compiling to stride-0 views:

  `rays_b = repeat(rays, 'r p d -> r t p d', t=NT)`     (insert t-axis)
  `tris_b = repeat(tris, 't v d -> r t v d', r=NR)`     (insert r-axis)

Both atoms are flavours of repeat: the first is plain `einops-repeat` adding a single axis, the second is `einops-repeat-broadcast` — same syntax, but the intent is to pair every-with-every. Together they're the canonical ARENA pattern for ray-triangle intersection batching.

### Composite Exercise — every-ray every-triangle (NR, NT, 3) via two repeats

**Atoms exercised together**: `einops-repeat`, `einops-repeat-broadcast`

Implement `cx7_pair_rays_with_triangles(rays, triangles)` that builds the every-ray every-triangle broadcast pair.

- `rays` has shape `(NR, 2, 3)` — origin and direction stacked along axis 1.
- `triangles` has shape `(NT, 3, 3)` — three vertices A,B,C stacked along axis 1.

1. **Repeat** rays into `(NR, NT, 2, 3)`: `repeat(rays, 'r p d -> r t p d', t=NT)`.
2. **Repeat-broadcast** triangles into `(NR, NT, 3, 3)`: `repeat(tris, 't v d -> r t v d', r=NR)`.

Return the tuple `(rays_b, tris_b)`. Both must be stride-0 views (zero-copy) — the test asserts `data_ptr` aliasing back to the original tensors.

In [ ]:
def cx7_pair_rays_with_triangles(rays, triangles):
    NR = rays.shape[0]
    NT = triangles.shape[0]
    # Atom A (einops-repeat): insert the t-axis on rays — stride-0 view.
    rays_b = repeat(rays, 'r p d -> r t p d', t=NT)
    # Atom B (einops-repeat-broadcast): insert the r-axis on triangles — stride-0 view.
    tris_b = repeat(triangles, 't v d -> r t v d', r=NR)
    return rays_b, tris_b


<details><summary>Show solution — cx7</summary>

```python
def cx7_pair_rays_with_triangles(rays, triangles):
    NR = rays.shape[0]
    NT = triangles.shape[0]
    # Atom A (einops-repeat): insert the t-axis on rays — stride-0 view.
    rays_b = repeat(rays, 'r p d -> r t p d', t=NT)
    # Atom B (einops-repeat-broadcast): insert the r-axis on triangles — stride-0 view.
    tris_b = repeat(triangles, 't v d -> r t v d', r=NR)
    return rays_b, tris_b
```

Both repeats compile to `expand` (stride-0 views) — memory stays O(NR + NT), not O(NR*NT). If you reach for `.repeat()` (the torch method, not einops), you allocate a full materialized (NR, NT, ...) tensor and lose the storage-aliasing property. The einops version is the only form that survives ARENA-scale (>>1M ray-tri pairs).
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx7'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx7',
        'subtopics': ["Einops: Repeat", "Einops: Repeat-as-broadcast"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()